# 06 — Tool Error Handling — Practical Examples

**Covers**: error classification, exponential backoff, graceful degradation, circuit breaker, logging

In [ ]:
import os, json, time, random, logging
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
print('✅ Ready')

## 1. Tools That Sometimes Fail (Simulated)

In [ ]:
import random

def flaky_search(query: str, failure_rate: float = 0.5) -> str:
    '''Simulates a tool that fails sometimes.'''
    if random.random() < failure_rate:
        raise ConnectionError('API timeout — search service unreachable')
    return f'Search results for "{query}": [Result 1, Result 2, Result 3]'

# Test without handling
for i in range(5):
    try:
        r = flaky_search('AI news', failure_rate=0.4)
        print(f'Call {i+1}: SUCCESS → {r[:50]}')
    except Exception as e:
        print(f'Call {i+1}: FAILED → {e}')

## 2. Exponential Backoff Retry

In [ ]:
def with_exponential_backoff(func, args, max_retries=4, base_delay=0.5):
    for attempt in range(max_retries):
        try:
            return func(**args)
        except (ConnectionError, TimeoutError) as e:
            if attempt == max_retries - 1:
                return f'ERROR_AFTER_RETRIES: {e}'
            delay = base_delay * (2 ** attempt) + random.uniform(0, 0.2)
            print(f'  Retry {attempt+1}/{max_retries} in {delay:.2f}s → {e}')
            time.sleep(delay)
        except Exception as e:
            return f'ERROR_NO_RETRY: {e}'
    return 'ERROR: Max retries exceeded'

print('Calling flaky_search with retry:')
result = with_exponential_backoff(flaky_search, {'query': 'Python 3.13', 'failure_rate': 0.5})
print(f'Final result: {result[:80]}')

## 3. Safe Tool Executor — Never Crashes the Loop

In [ ]:
TOOL_MAP = {
    'web_search': lambda query: flaky_search(query, 0.3),
    'calculator': lambda expression: str(eval(expression, {'__builtins__': {}}, {})),
}

def safe_execute(tool_name: str, args: dict) -> str:
    '''Execute a tool safely — NEVER raises, always returns string for LLM.'''
    if tool_name not in TOOL_MAP:
        return f'ERROR: Tool "{tool_name}" not in registry. Available: {list(TOOL_MAP.keys())}'
    try:
        raw = args
        result = with_exponential_backoff(TOOL_MAP[tool_name], raw, max_retries=3)
        # Truncate if too long
        if len(str(result)) > 3000:
            result = str(result)[:3000] + '... [truncated]'
        return str(result)
    except Exception as e:
        return f'ERROR_UNEXPECTED: {e}'

# Test safe executor
test_calls = [
    ('web_search', {'query': 'AI 2025'}),
    ('calculator', {'expression': '2 ** 10'}),
    ('unknown_tool', {'x': 1}),
    ('calculator', {'expression': '1/0'}),
]
for name, args in test_calls:
    r = safe_execute(name, args)
    status = '✅' if not r.startswith('ERROR') else '❌'
    print(f'{status} {name}({args}) → {r[:60]}')

## 4. Circuit Breaker

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold=3, cooldown=5):
        self.failures = 0
        self.threshold = threshold
        self.cooldown = cooldown
        self.last_fail_time = None
        self.state = 'CLOSED'
    
    def can_call(self):
        if self.state == 'OPEN':
            if time.time() - self.last_fail_time > self.cooldown:
                self.state = 'HALF_OPEN'
                return True
            return False
        return True
    
    def record_success(self):
        self.failures = 0
        self.state = 'CLOSED'
    
    def record_failure(self):
        self.failures += 1
        self.last_fail_time = time.time()
        if self.failures >= self.threshold:
            self.state = 'OPEN'
            print(f'  ⚡ Circuit OPEN — tool disabled for {self.cooldown}s')

cb = CircuitBreaker(threshold=3, cooldown=2)

for i in range(7):
    if not cb.can_call():
        print(f'Call {i+1}: BLOCKED by circuit breaker (state={cb.state})')
        continue
    try:
        result = flaky_search('test', failure_rate=0.7)
        cb.record_success()
        print(f'Call {i+1}: ✅ SUCCESS')
    except Exception as e:
        cb.record_failure()
        print(f'Call {i+1}: ❌ FAILED → {e}')